In [ ]:
import nltk
nltk.download('movie_reviews')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


In [4]:
from nltk.corpus import movie_reviews
movie_reviews.fileids()[:10]  #문서아이디
movie_reviews.categories()
print( len(movie_reviews.fileids(categories='neg')) )
print( len(movie_reviews.fileids(categories='pos')) )

1000
1000


In [5]:
fieldid = movie_reviews.fileids()[0]
print(movie_reviews.raw(fieldid)[:100])
print( movie_reviews.categories(fieldid) )

plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 

['neg']


In [6]:
ids = movie_reviews.fileids()
reviews = [movie_reviews.raw(id) for id in ids]
categories = [movie_reviews.categories(id)[0]  for id in ids]

### TextBlob을 이용한 감성 분석

1: https://textblob.readthedocs.io/en/dev/

https://textblob.readthedocs.io/en/dev/quickstart.html

In [ ]:
# 단어별 감성점수의 평균을 구하고, 강조어(very, extremely)나 부정어(not)에대한 처리규칙이 포함되어 있음

'plot : two'

In [19]:
%pip install textblob

   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 625.0/625.0 kB 22.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [22]:
!python -m textblob.download_corpora

Finished.


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\brown.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\conll2000.zip.
[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


In [7]:
from textblob import TextBlob
result = TextBlob('the sky is blue')
result.sentiment
# polarity  -1 ~ 1  0 중립  1이면 매우긍정
# subjectivity 0 ~ 1  1: 매우 주관적  0 : 객관적

Sentiment(polarity=0.0, subjectivity=0.1)

In [8]:
from sklearn.metrics import accuracy_score
predict = [TextBlob(review).sentiment for review in reviews]

In [21]:
import numpy as np
textblob_predict = [pred[0] for pred in predict]
textblob_predict = np.array(textblob_predict) > 0.1
textblob_predict = ['pos' if pred == True else 'neg' for pred in textblob_predict ]
accuracy_score(categories,textblob_predict)

0.746

In [22]:
# 특징 : 주관성 지수를 함께 제공
# 리뷰중에서 주관적 과 객관적의 비율

### AFINN을 이용한 감성 분석

https://github.com/fnielsen/afinn 

(1) http://corpustext.com/reference/sentiment_afinn.html

In [23]:
# 감성점수를 -5 ~ +5 점수체계 : 처리속도가 빠름
%pip install afinn

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for afinn: filename=afinn-0.1-py3-none-any.whl size=53479 sha256=e5d2f259de98b4d54e51158624be0bcfd3431e97904851bc3eab640da4b0d6cc
  Stored in directory: c:\users\playdata\appdata\local\pip\cache\wheels\b0\05\90\43f79196199a138fb486902fceca30a2d1b5228e6d2db8eb90
Successfully built afinn
Note: you may need to restart the kernel to use updated packages.


In [24]:
from afinn import Afinn

In [29]:
afn = Afinn(emoticons=True)
result = np.array([afn.score(review) for review in reviews]) > 0
result = ['pos' if re == True else 'neg'  for re in result]
accuracy_score(categories, result)

0.664

### VADER를 이용한 감성 분석

(1) https://github.com/cjhutto/vaderSentiment

In [30]:
# 대문자 구두점 이모티콘 sns 데이터 특유의 감성표현을포착하는 '문법적 휴리스특'원리를 기술

In [31]:
%pip install vaderSentiment

Note: you may need to restart the kernel to use updated packages.


In [32]:
%pip install --upgrade vaderSentiment

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm import tqdm
analyzer = SentimentIntensityAnalyzer()
result = []
for sentence in tqdm(reviews):
    vs = analyzer.polarity_scores(sentence)
    # print("{:-<65} {}".format(sentence[:10], str(vs)))
    if vs['compound'] > 0:
        result.append('pos')
    else:
        result.append('neg')    

plot : two------------------------------------------------------- {'neg': 0.092, 'neu': 0.802, 'pos': 0.106, 'compound': 0.9407}
the happy ------------------------------------------------------- {'neg': 0.034, 'neu': 0.892, 'pos': 0.074, 'compound': 0.8996}
it is movi------------------------------------------------------- {'neg': 0.086, 'neu': 0.816, 'pos': 0.098, 'compound': 0.6368}
 " quest f------------------------------------------------------- {'neg': 0.131, 'neu': 0.748, 'pos': 0.12, 'compound': -0.8144}
synopsis :------------------------------------------------------- {'neg': 0.065, 'neu': 0.869, 'pos': 0.066, 'compound': 0.6594}
capsule : ------------------------------------------------------- {'neg': 0.133, 'neu': 0.792, 'pos': 0.076, 'compound': -0.9939}
so ask you------------------------------------------------------- {'neg': 0.098, 'neu': 0.833, 'pos': 0.069, 'compound': -0.9724}
that's exa------------------------------------------------------- {'neg': 0.13, 'neu': 0.715, '